# 🔍 What is RAG?

## The Simple Idea

**RAG = give the LLM a textbook to look at before answering.**

Instead of relying on memory alone, the LLM gets to look up relevant information first.

```
WITHOUT RAG:
  You: "What is our refund policy?"
  LLM: (guesses from training data) "Most companies offer 30-day refunds..."
       ← could be wrong!

WITH RAG:
  You: "What is our refund policy?"
   ↓
  System searches your documents...
  Found: "Our policy.pdf: All purchases are refundable within 14 days."
   ↓
  LLM: "According to your policy, all purchases are refundable within 14 days."
       ← grounded in real data!
```

## The 3 Steps of RAG

```
Step 1 — RETRIEVE
  Search your documents for the most relevant chunks

Step 2 — AUGMENT
  Add those chunks to the prompt as context

Step 3 — GENERATE
  LLM reads the context and writes the answer
```

That's where the name comes from: **R**etrieval **A**ugmented **G**eneration.

In [9]:
# The simplest possible RAG — no libraries, just Python

# Your "knowledge base" — could be docs, FAQs, a manual, anything
knowledge_base = [
    "Our refund policy allows returns within 14 days of purchase.",
    "Shipping takes 3-5 business days for standard delivery.",
    "Premium members get free shipping on all orders.",
    "Customer support is available Monday to Friday, 9am-5pm.",
    "You can track your order on the website under 'My Orders'.",
]

print(f"Knowledge base has {len(knowledge_base)} documents")

Knowledge base has 5 documents


In [10]:
# Step 1: RETRIEVE — find relevant documents
# (Using keyword search here — we'll learn better methods in later sections)

def simple_retrieve(question, docs, top_k=2):
    question_words = set(question.lower().split())
    scored = []
    for doc in docs:
        doc_words = set(doc.lower().split())
        overlap = len(question_words & doc_words)  # count matching words
        scored.append((overlap, doc))
    scored.sort(reverse=True)
    return [doc for _, doc in scored[:top_k]]

question = "How long do I have to return something?"
relevant_docs = simple_retrieve(question, knowledge_base)

print(f"Question: '{question}'")
print("\nRetrieved documents:")
for doc in relevant_docs:
    print(f"  → {doc}")

Question: 'How long do I have to return something?'

Retrieved documents:
  → Customer support is available Monday to Friday, 9am-5pm.
  → You can track your order on the website under 'My Orders'.


In [11]:
# Step 2: AUGMENT — add context to the prompt

context = "\n".join(relevant_docs)

prompt = f"""Use the information below to answer the question.

Information:
{context}

Question: {question}
Answer:"""

print("Prompt sent to LLM:")
print("-" * 40)
print(prompt)

Prompt sent to LLM:
----------------------------------------
Use the information below to answer the question.

Information:
Customer support is available Monday to Friday, 9am-5pm.
You can track your order on the website under 'My Orders'.

Question: How long do I have to return something?
Answer:


In [12]:
# Step 3: GENERATE — LLM reads context and answers
# (Simulated here — swap with a real API call)

# Real version:
# from anthropic import Anthropic
# client = Anthropic()
# response = client.messages.create(
#     model="claude-sonnet-4-6", max_tokens=200,
#     messages=[{"role": "user", "content": prompt}]
# )
# answer = response.content[0].text

answer = "You have 14 days from the date of purchase to return something."

print(f"Question: {question}")
print(f"Answer:   {answer}")
print()
print("✅ The answer came from YOUR documents, not the LLM's training memory.")
print("   That's RAG!")

Question: How long do I have to return something?
Answer:   You have 14 days from the date of purchase to return something.

✅ The answer came from YOUR documents, not the LLM's training memory.
   That's RAG!


## Why RAG vs Fine-tuning?

```
Fine-tuning:  Bake knowledge INTO the model weights
              → Expensive, slow, can't update easily

RAG:          Look up knowledge AT QUERY TIME
              → Cheap, fast, update by changing your docs
```

**Use RAG when:**
- Your knowledge changes frequently
- You need to cite sources
- Your data is private / proprietary
- You want to avoid hallucination

---
Next: `02_Your_First_RAG.ipynb` — build a complete RAG system in under 20 lines.